In [8]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import classification_report

# 1. Hardware Optimization
torch.set_num_threads(12)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hardware: {device} | Threads CPU: {torch.get_num_threads()}")

# 2. Setup Path
path_weights = r'D:\S2\thesis\cond\project_ids\models\Smart_SLGRAE_FINAL_PRO.pth'
path_scaler = r'D:\S2\thesis\cond\project_ids\data\processed\scaler.pkl'
path_pkl = r'D:\S2\thesis\cond\project_ids\data\processed\split_balanced_data.pkl'
path_test_csv = r'D:\S2\thesis\Conda\CICIOT23\test\test.csv'

# 3. Arsitektur (Sesuai Notebook 06)
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
    def forward(self, x): return torch.relu(x + self.net(x))

class Smart_SLGRAE_Latent32(nn.Module):
    def __init__(self, input_dim=44, latent_dim=32, num_classes=34):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(), ResidualBlock(128), nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.4), ResidualBlock(256), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
        self.log_var_cls = nn.Parameter(torch.zeros(1))
        self.log_var_rec = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        latent = self.encoder(x)
        return self.classifier(latent), self.decoder(latent), latent

# 4. Load Model & Tools
model = Smart_SLGRAE_Latent32().to(device)
checkpoint = torch.load(path_weights, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

scaler = joblib.load(path_scaler)
data_info = joblib.load(path_pkl)
class_names = data_info['classes']
label_mapping = {name: i for i, name in enumerate(class_names)}

# KUNCI: Ambil urutan kolom asli dari training
# Notebook 01 lo drop 'label', urutan sisanya adalah ini:
ordered_features = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 
    'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count', 
    'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'ICMP', 
    'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 
    'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight'
]

# 5. Batch Processing
y_true_all, y_pred_all = [], []
chunk_size = 100000 

print(f"⏳ Memproses {path_test_csv}...")
reader = pd.read_csv(path_test_csv, chunksize=chunk_size)

for i, chunk in enumerate(reader):
    # Paksa urutan kolom sinkron dengan scaler
    X_raw = chunk[ordered_features].values
    y_str = chunk['label'].values
    
    # Scaling & Prediksi GPU
    X_scaled = scaler.transform(X_raw)
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    
    with torch.no_grad():
        logits, _, _ = model(X_tensor)
        _, preds = torch.max(logits, 1)
    
    y_true_all.extend([label_mapping.get(str(l), 0) for l in y_str])
    y_pred_all.extend(preds.cpu().numpy())
    print(f"🔄 Batch {i+1} selesai...")

print("\n" + "="*60)
print("📊 FINAL VALIDATION REPORT")
print("="*60)
print(classification_report(y_true_all, y_pred_all, target_names=class_names))

🚀 Hardware: cuda | Threads CPU: 12


C:\Users\Z\AppData\Local\Temp\ipykernel_2804\1913495994.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path_weights, map_location=device)


⏳ Memproses D:\S2\thesis\Conda\CICIOT23\test\test.csv...


c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


🔄 Batch 1 selesai...


...

c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


🔄 Batch 12 selesai...

📊 FINAL VALIDATION REPORT


c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                         precision    recall  f1-score   support

       Backdoor_Malware       0.14      0.07      0.09        89
          BenignTraffic       0.80      0.04      0.08     27709
       BrowserHijacking       0.04      0.01      0.02       134
       CommandInjection       0.09      0.05      0.06       119
 DDoS-ACK_Fragmentation       1.00      0.74      0.85      7292
        DDoS-HTTP_Flood       0.00      0.00      0.00       709
        DDoS-ICMP_Flood       1.00      1.00      1.00    180447
DDoS-ICMP_Fragmentation       0.99      0.98      0.99     11402
      DDoS-PSHACK_Flood       0.00      0.00      0.00    103326
       DDoS-RSTFINFlood       0.00      0.00      0.00    101819
         DDoS-SYN_Flood       0.00      0.00      0.00    102208
         DDoS-SlowLoris       0.05      0.00      0.00       622
DDoS-SynonymousIP_Flood       0.03      0.00      0.00     90480
         DDoS-TCP_Flood       0.00      0.00      0.00    113735
         DDoS-UDP_Flood 

c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Z\miniconda3\envs\tesis\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
